# Notebook 08 - Actividad Final: Asistente Documental Inteligente

## Objetivos
- Integrar sentimiento BERT, QA y resumen GPT.
- Procesar `documentos_qa.csv` y textos personalizados.
- Documentar configuracion, resultados y mejoras futuras.

## Introduccion
Proyecto integrador resuelto: un pipeline que analiza documentos, responde preguntas, estima tono y genera resumenes ejecutivos.

In [1]:
from pathlib import Path
from dataclasses import dataclass
from IPython.display import display
import pandas as pd
from transformers import pipeline, set_seed

set_seed(42)
RUTA_QA = Path('..') / 'datasets' / 'documentos_qa.csv'
print('Iniciando Asistente Documental Inteligente')

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Iniciando Asistente Documental Inteligente


## 1) Configuracion del pipeline

In [ ]:
@dataclass
class ConfigAsistente:
    modelo_sentimiento: str = "pysentimiento/robertuito-sentiment-analysis"
    # Question Answering
    modelo_qa: str = "PlanTL-GOB-ES/roberta-base-bne-sqac"
    # Generación de texto
    modelo_generacion: str = "datificate/gpt2-small-spanish"
    max_tokens_resumen: int = 60
    temperatura: float = 0.6

cfg = ConfigAsistente()
print(cfg)

ConfigAsistente(modelo_sentimiento='distilbert-base-uncased-finetuned-sst-2-english', modelo_qa='distilbert-base-cased-distilled-squad', modelo_generacion='gpt2', max_tokens_resumen=60, temperatura=0.6)


## 2) Inicializar componentes

In [3]:
# Cargamos tres pipelines especializados
sentiment_pipe = pipeline('sentiment-analysis', model=cfg.modelo_sentimiento)
qa_pipe = pipeline('question-answering', model=cfg.modelo_qa)
gen_pipe = pipeline('text-generation', model=cfg.modelo_generacion)
print('Componentes listos: sentimiento, QA, generacion')

Device set to use cpu
Device set to use cpu
Device set to use cpu


Componentes listos: sentimiento, QA, generacion


## 3) Funciones del asistente

In [4]:
def analizar_sentimiento(texto_en: str) -> dict:
    out = sentiment_pipe(texto_en[:512])[0]
    return {'label': out['label'], 'score': round(out['score'], 4)}

def responder_pregunta(contexto: str, pregunta: str) -> dict:
    out = qa_pipe(question=pregunta, context=contexto)
    return {'answer': out['answer'], 'score': round(out['score'], 4)}

def resumir_documento(texto: str) -> str:
    prompt = f'Summarize clearly:\n{texto}\nSummary:'
    out = gen_pipe(
        prompt,
        max_new_tokens=cfg.max_tokens_resumen,
        do_sample=True,
        temperature=cfg.temperatura,
    )
    return out[0]['generated_text']

print('Funciones definidas correctamente')

Funciones definidas correctamente


## 4) Procesar documentos_qa.csv

In [5]:
df = pd.read_csv(RUTA_QA)
resultados = []

for _, row in df.iterrows():
    contexto = row['contexto']
    pregunta = row['pregunta']
    qa = responder_pregunta(contexto, pregunta)
    sent = analizar_sentimiento(contexto)
    resumen = resumir_documento(contexto)
    resultados.append({
        'pregunta': pregunta,
        'respuesta_esperada': row['respuesta'],
        'respuesta_modelo': qa['answer'],
        'qa_score': qa['score'],
        'sentimiento': sent['label'],
        'sentimiento_score': sent['score'],
        'resumen': resumen,
    })

df_resultados = pd.DataFrame(resultados)
display(df_resultados)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,pregunta,respuesta_esperada,respuesta_modelo,qa_score,sentimiento,sentimiento_score,resumen
0,¿Quién creó Python?,Guido van Rossum,Python fue creado por Guido van Rossum y publi...,0.1279,NEGATIVE,0.9901,Summarize clearly:\nPython fue creado por Guid...
1,¿En qué año se publicó Python?,1991,1991,0.1234,NEGATIVE,0.9901,Summarize clearly:\nPython fue creado por Guid...
2,¿Qué empresa introdujo los Transformers?,Google,Google,0.0967,NEGATIVE,0.8632,Summarize clearly:\nLos Transformers fueron in...
3,¿En qué año se publicó el paper?,2017,Los Transformers fueron introducidos,0.6124,NEGATIVE,0.8632,Summarize clearly:\nLos Transformers fueron in...
4,¿Qué tipo de arquitectura usa BERT?,encoder-only,con Masked Language Modeling y Next Sentence P...,0.0386,NEGATIVE,0.9971,Summarize clearly:\nBERT es un modelo encoder-...
5,¿Qué predice GPT?,la siguiente palabra,only,0.0328,NEGATIVE,0.9944,Summarize clearly:\nGPT predice la siguiente p...
6,¿Qué permite la Self-Attention?,que cada token observe a todos los demás tokens,La capa de Self-Attention permite que cada token,0.3655,NEGATIVE,0.9944,Summarize clearly:\nLa capa de Self-Attention ...
7,¿Cómo reformula T5 las tareas?,texto a texto,NLP como problemas de texto a texto,0.1735,NEGATIVE,0.9947,Summarize clearly:\nT5 reformula todas las tar...


## 5) Texto personalizado del usuario

In [6]:
texto_custom = (
    'Our support team resolved 95% of tickets within 24 hours. '
    'Customers reported high satisfaction after the new chatbot launch.'
)
pregunta_custom = 'What percentage of tickets were resolved within 24 hours?'

reporte = {
    'sentimiento': analizar_sentimiento(texto_custom),
    'respuesta': responder_pregunta(texto_custom, pregunta_custom),
    'resumen': resumir_documento(texto_custom),
}
display(pd.DataFrame([reporte]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,sentimiento,respuesta,resumen
0,"{'label': 'POSITIVE', 'score': 0.9974}","{'answer': '95%', 'score': 0.9463}",Summarize clearly:\nOur support team resolved ...


## 6) Metricas agregadas del proyecto

In [7]:
aciertos_qa = (df_resultados['respuesta_modelo'].str.lower().str.strip()
               == df_resultados['respuesta_esperada'].str.lower().str.strip())
print('Exact match QA:', f"{aciertos_qa.mean():.1%}")
print('Sentimiento positivo promedio:',
      (df_resultados['sentimiento'] == 'POSITIVE').mean())

Exact match QA: 25.0%
Sentimiento positivo promedio: 0.0


## 7) Mejoras propuestas
- Usar modelos multilingues para documentos en espanol.
- Agregar chunking para contextos largos.
- Evaluar con RAG sobre base vectorial corporativa.
- Registrar trazas y scores en un dashboard de monitoreo.

## Resultados
El asistente combino tres capacidades sobre CSV y texto libre, con metricas basicas de exactitud QA y tono del documento.

## Conclusiones
Un sistema documental util une comprension (BERT), extraccion (QA) y sintesis (GPT). La calidad mejora con datos del dominio y evaluacion continua.

## Ejercicios guiados resueltos
**Ejercicio:** Crea funcion `procesar_documento(texto, preguntas)` que devuelva JSON.

**Solucion:**

In [8]:
def procesar_documento(texto: str, preguntas: list[str]) -> dict:
    return {
        'sentimiento': analizar_sentimiento(texto),
        'resumen': resumir_documento(texto),
        'qa': [responder_pregunta(texto, p) for p in preguntas],
    }

ejemplo = procesar_documento(texto_custom, [pregunta_custom])
print(list(ejemplo.keys()))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['sentimiento', 'resumen', 'qa']


## Ejercicios propuestos
1. Empaqueta el asistente como API FastAPI.
2. Agrega validacion de entrada y limites de tokens.
3. Disena pruebas unitarias para `responder_pregunta`.

## Preguntas de reflexion
1. Que componente fallaria primero en documentos de 50 paginas?
2. Como combinarias este pipeline con embeddings de Clase 01?
3. Que metricas de negocio reportarias ademas de exact match?